### ロード

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import skew
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
# --- ファイル読み込み ---

df_raw = pd.read_parquet("liq_engine_raw.parquet", engine="pyarrow")
df_feats_d = pd.read_parquet("liq_engine_feats_d.parquet", engine="pyarrow")
df_feats_w = pd.read_parquet("liq_engine_feats_w.parquet", engine="pyarrow")
df_feats_w.columns

### 特徴量性格診断

In [ ]:
# 特徴量生成
df_feats_w["DFII10_roc4"] = df_feats_w["DFII10"].diff(4)

In [ ]:
# リターン生成
lag_weeks = 0
forward_weeks = 8
target_return = f'lag_{lag_weeks}weeks_return_{forward_weeks}weeks'
df_feats_w[target_return] = (
    (df_feats_w["^GSPC"].shift(-(lag_weeks + forward_weeks))) / 
     df_feats_w["^GSPC"].shift(-lag_weeks) -1
)

In [ ]:
# stats可視化 median IQR cvar_5
feature = "PERMIT_GOLD_ratio_z52"
is_inverse = False

df = df_feats_w[[feature, target_return]].copy()
df = df.loc["2010-01-01":]
# ランクの作成
df[f'{feature}_rank'] = pd.qcut(df[feature].dropna(), 5, labels=False, duplicates='drop') + 1
if is_inverse:
    df[f'{feature}_rank'] = 6 - df[f'{feature}_rank'] # 1が最悪(重力)、5が最高(浮力)に統一

# クロス集計を作成（行：現在、列：次期）
df['next_rank'] = df[f'{feature}_rank'].shift(-1)
transition_matrix = pd.crosstab(
    df[f'{feature}_rank'], 
    df['next_rank'], 
    normalize='index'
)
# 各統計量の計算
stats = []
for r in range(1, 6):
    d = df[df[f'{feature}_rank'] == r][target_return]
    stats.append({
        'rank': r,
        'median': d.median(),
        'iqr': d.quantile(0.75) - d.quantile(0.25),
        'cvar_5': d[d <= d.quantile(0.05)].mean(),
        'win_rate': (d > 0).mean()
    })
s_df = pd.DataFrame(stats)

fig = make_subplots(
    rows=2, cols=3, 
    subplot_titles=('IQR: Lower is better', 'Median', 'Tail Risk (CVaR 5%)', "Win_Rate", 'Transition Probability (Next Week)'),
    vertical_spacing=0.1,
    specs=[[{"type": "bar"}, {"type": "scatter"}, {"type": "bar"}],
           [{"type": "scatter"}, {"type": "heatmap", "colspan": 2}, None]] # ヒートマップを2列分使う
)

fig.add_trace(go.Bar(x=s_df['rank'], y=s_df['iqr'], name='IQR', marker_color='lightblue'), row=1, col=1)
fig.add_trace(go.Scatter(x=s_df['rank'], y=s_df['median'], name='Median', mode='lines+markers', line_color='orange'), row=1, col=2)
fig.add_trace(go.Bar(x=s_df['rank'], y=s_df['cvar_5'], name='CVaR', marker_color='crimson'), row=1, col=3)
fig.add_trace(go.Scatter(x=s_df['rank'], y=s_df['win_rate'], name='Win Rate', marker_color='crimson'), row=2, col=1)
# 遷移ヒートマップの追加
fig.add_trace(
    go.Heatmap(
        z=transition_matrix.values,
        x=[f"Next R{i}" for i in transition_matrix.columns],
        y=[f"Now R{i}" for i in transition_matrix.index],
        colorscale='Viridis',
        text=np.around(transition_matrix.values, 2), # 数値を表示
        texttemplate="%{text}",
        showscale=False
    ),
    row=2, col=2
)
fig.update_layout(
    title_text=f"Personality Profile: {feature} - Lag:{lag_weeks} weeks, Return:forward {forward_weeks} weeks",
    showlegend=False, template="plotly_dark"
)
fig.show(renderer="browser")


In [ ]:
# 箱ひげ図の描画
plot_df = df.dropna(subset=[target_return, f'{feature}_rank'])

rank_name = f'{feature}_rank'
fig = px.box(
    plot_df,
    x=rank_name,
    y=target_return,
    color=rank_name,
    points="all", # 全データ点（ジッター）を表示して密度を確認
    title=f"Market Physics: S&P 500 {forward_weeks}W Forward Return by {feature}_rank",
    labels={target_return: f'{forward_weeks}-Week Forward Return', '{feature}_rank': 'Liquidity Regime'},
    category_orders={f"{feature}_rank": ['1: Very Low (Heavy)', '2: Low', '3: Neutral', '4: High', '5: Very High (Buoyancy)']}
)

# 0%のライン（損益分岐）を強調
fig.add_hline(y=0, line_dash="dash", line_color="black", annotation_text="Break Even")

# レイアウト調整
fig.update_layout(
    xaxis_title="Liquidity Regime (Liq_eff Z-score)",
    yaxis_title=f"Forward {forward_weeks}W Return (%)",
    showlegend=False,
    template="plotly_white",
    yaxis=dict(tickformat=".1%")
)

# 表示
fig.show(renderer="browser")

### マクロ潮流

In [ ]:
df = df_raw[["credit_gap", "dsr", "DFII10", "T10YIE", "WALCL"]].dropna(how="all")
def zscore(df):
    return (df - df.mean()) / df.std()

credit_gap = df["credit_gap"].dropna()
credit_gap = zscore(credit_gap)
dsr = df["dsr"].dropna()
dsr = zscore(dsr)
DFII10 = df["DFII10"].dropna()
DFII10 = zscore(DFII10)
T10YIE = df["T10YIE"].dropna()
T10YIE = zscore(T10YIE)
WALCL = df["WALCL"].dropna()
WALCL = zscore(WALCL)


In [37]:
fig = go.Figure()

col = ["credit_gap", "dsr", "DFII10", "T10YIE", "WALCL"]
for col in df.columns:
    df_sub = df[col].dropna()
    fig.add_trace(go.Scatter(
        x=df_sub.index,
        y=df_sub,
        mode='lines',
        name=col,
    ))
fig.show(renderer="browser")

### 

### 